In [1]:
import pandas as pd
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [2]:
notes_val = pd.read_csv("notes_val_predictions_task2.csv")
ecg_val = pd.read_csv("ecg_val_predictions_task2.csv")
structured_val = pd.read_csv("structured_val_predictions_task2.csv")

display(notes_val.head())
display(ecg_val.head())
display(structured_val.head())

print(len(notes_val))
print(len(ecg_val))
print(len(structured_val))

,sample_id,y_true,pred_prob_notes
0,0,1,0.083591
1,1,0,0.408019
2,2,0,0.030880
3,3,0,0.046335
4,4,0,0.041462


,sample_id,y_true,pred_prob_ecg
0,0,1,0.221242
1,1,0,0.074797
2,2,0,0.089479
3,3,0,0.075146
4,4,0,0.075561


,sample_id,y_true,pred_prob_structured
0,0,1,0.601427
1,1,0,0.036917
2,2,0,0.041250
3,3,0,0.205017
4,4,0,0.011207


16262
16262
16262


In [3]:
val_df = notes_val.merge(
    ecg_val[["sample_id", "pred_prob_ecg"]],
    on="sample_id",
    how="inner"
).merge(
    structured_val[["sample_id", "pred_prob_structured"]],
    on="sample_id",
    how="inner"
)

val_df

,sample_id,y_true,pred_prob_notes,pred_prob_ecg,pred_prob_structured
0,0,1,0.083591,0.221242,0.601427
1,1,0,0.408019,0.074797,0.036917
2,2,0,0.030880,0.089479,0.041250
3,3,0,0.046335,0.075146,0.205017
4,4,0,0.041462,0.075561,0.011207
...,...,...,...,...,...
16257,16257,0,0.355216,0.075690,0.201318
16258,16258,0,0.046120,0.080014,0.011143
16259,16259,1,0.076822,0.083144,0.227490
16260,16260,0,0.029619,0.074411,0.025802


In [4]:
notes_test = pd.read_csv("notes_test_predictions_task2.csv")
ecg_test = pd.read_csv("ecg_test_predictions_task2.csv")
structured_test = pd.read_csv("structured_test_predictions_task2.csv")

display(notes_test.head())
display(ecg_test.head())
display(structured_test.head())

print(len(notes_test))
print(len(ecg_test))
print(len(structured_test))

,sample_id,y_true,pred_prob_notes
0,0,0,0.287968
1,1,0,0.029424
2,2,0,0.097648
3,3,0,0.034521
4,4,0,0.271976


,sample_id,y_true,pred_prob_ecg
0,0,0,0.076260
1,1,0,0.075224
2,2,0,0.074155
3,3,0,0.075142
4,4,0,0.081039


,sample_id,y_true,pred_prob_structured
0,0,0,0.017748
1,1,0,0.017309
2,2,0,0.009830
3,3,0,0.014482
4,4,0,0.075333


16262
16262
16262


In [5]:
test_df = notes_test.merge(
    ecg_test[["sample_id", "pred_prob_ecg"]],
    on="sample_id",
    how="inner"
).merge(
    structured_test[["sample_id", "pred_prob_structured"]],
    on="sample_id",
    how="inner"
)
test_df

,sample_id,y_true,pred_prob_notes,pred_prob_ecg,pred_prob_structured
0,0,0,0.287968,0.076260,0.017748
1,1,0,0.029424,0.075224,0.017309
2,2,0,0.097648,0.074155,0.009830
3,3,0,0.034521,0.075142,0.014482
4,4,0,0.271976,0.081039,0.075333
...,...,...,...,...,...
16257,16257,0,0.071461,0.075118,0.018742
16258,16258,0,0.107719,0.127407,0.075378
16259,16259,0,0.031589,0.074343,0.018229
16260,16260,0,0.055875,0.086359,0.095523


In [6]:
# Feature engineering
def add_meta_features(df):
    eps = 1e-6
    
    # base probs
    s = df["pred_prob_structured"].astype(float)
    n = df["pred_prob_notes"].astype(float)
    e = df["pred_prob_ecg"].astype(float)
    
    # interaction terms
    df["notes_x_ecg"] = n * e
    df["notes_x_struct"] = n * s
    df["ecg_x_struct"] = e * s
    df["notes_x_ecg_x_struct"] = n * e * s
    
    # pairwise differences
    df["struct_minus_notes"] = s - n
    df["struct_minus_ecg"] = s - e
    df["notes_minus_ecg"] = n - e
    
    # summary stats across modalities
    df["prob_mean"] = (s + n + e) / 3.0
    df["prob_max"] = np.maximum(np.maximum(s, n), e)
    df["prob_min"] = np.minimum(np.minimum(s, n), e)
    df["prob_range"] = df["prob_max"] - df["prob_min"]
    
    # agreement / dispersion
    df["prob_std"] = np.std(np.vstack([s, n, e]), axis=0)
    
    # logit transform can help logistic stacking
    df["logit_struct"] = np.log((s + eps) / (1 - s + eps))
    df["logit_notes"] = np.log((n + eps) / (1 - n + eps))
    df["logit_ecg"] = np.log((e + eps) / (1 - e + eps))
    
    # rank-like indicators / thresholds
    df["struct_gt_notes"] = (s > n).astype(int)
    df["struct_gt_ecg"] = (s > e).astype(int)
    df["notes_gt_ecg"] = (n > e).astype(int)
    
    # confidence-style features
    df["struct_conf"] = np.abs(s - 0.5)
    df["notes_conf"] = np.abs(n - 0.5)
    df["ecg_conf"] = np.abs(e - 0.5)
    
    return df

val_df = add_meta_features(val_df.copy())
test_df = add_meta_features(test_df.copy())

# Features
stack_features = [
    # raw probs
    "pred_prob_structured",
    "pred_prob_notes",
    "pred_prob_ecg",
    
    # interactions
    "notes_x_ecg",
    "notes_x_struct",
    "ecg_x_struct",
    "notes_x_ecg_x_struct",
    
    # differences
    "struct_minus_notes",
    "struct_minus_ecg",
    "notes_minus_ecg",
    
    # summaries
    "prob_mean",
    "prob_max",
    "prob_min",
    "prob_range",
    "prob_std",
    
    # logits
    "logit_struct",
    "logit_notes",
    "logit_ecg",
    
    # comparisons
    "struct_gt_notes",
    "struct_gt_ecg",
    "notes_gt_ecg",
    
    # confidence
    "struct_conf",
    "notes_conf",
    "ecg_conf",
]

X_val = val_df[stack_features].replace([np.inf, -np.inf], np.nan).fillna(0)
y_val = val_df["y_true"].astype(int)

X_test = test_df[stack_features].replace([np.inf, -np.inf], np.nan).fillna(0)
y_test = test_df["y_true"].astype(int)

# Logistic stacking model
meta_model = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        l1_ratio=0.2,
        C=0.5,
        max_iter=5000,
        class_weight="balanced",
        random_state=42
    ))
])

meta_model.fit(X_val, y_val)

# Predict + evaluate
test_df["pred_logistic_ensemble"] = meta_model.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, test_df["pred_logistic_ensemble"])
auprc = average_precision_score(y_test, test_df["pred_logistic_ensemble"])

print("Logistic multimodal test AUROC:", round(auc, 6))
print("Logistic multimodal test AUPRC:", round(auprc, 6))

# Inspect learned weights
lr = meta_model.named_steps["lr"]
coefs = pd.DataFrame({
    "feature": stack_features,
    "coef": lr.coef_[0]
}).sort_values("coef", ascending=False)

print("\nTop positive coefficients:")
print(coefs.head(10).to_string(index=False))

print("\nTop negative coefficients:")
print(coefs.tail(10).to_string(index=False))

Logistic multimodal test AUROC: 0.873248
Logistic multimodal test AUPRC: 0.430102

Top positive coefficients:
             feature         coef
        logit_struct 1.997676e+00
         logit_notes 1.077695e+00
            prob_std 3.609439e-01
      notes_x_struct 1.231792e-01
notes_x_ecg_x_struct 9.271373e-02
          notes_conf 4.883664e-02
        ecg_x_struct 1.094508e-02
            ecg_conf 9.223531e-03
       struct_gt_ecg 8.084479e-04
           logit_ecg 1.613762e-08

Top negative coefficients:
             feature      coef
        notes_gt_ecg -0.036652
         struct_conf -0.044226
     struct_gt_notes -0.059140
         notes_x_ecg -0.113683
            prob_min -0.118330
    struct_minus_ecg -0.311806
pred_prob_structured -0.314312
     pred_prob_notes -0.331349
     notes_minus_ecg -0.331993
           prob_mean -0.400027
